# AniList Dataset Retrieval

## Manga-adapted vs. Light-Novel-adapted TV Anime Ratings

This notebook retrieves the dataset used for the **Choose Your Own Hypothesis (CYOH)** project directly from the AniList public GraphQL API.

### Population definition
- Anime
- TV format
- Finished series
- Non-adult titles
- Start date from **2015-01-01 through 2025-12-31**
- Source material classified as **MANGA** or **LIGHT_NOVEL**

### Outputs
1. `anilist_raw_manga_vs_light_novel_2015_2025.csv`
2. `anilist_analysis_manga_vs_light_novel_2015_2025.csv`

The raw dataset contains all qualifying records returned by the API. The analysis dataset removes only records where `average_score` is missing and adds a readable `source_group` label.

## 1. Imports and API settings

This retrieval notebook uses only Python standard-library modules, so no third-party packages are required.

In [ ]:
import csv
import json
import time
import urllib.request
import urllib.error

API_URL = "https://graphql.anilist.co"
START_BOUND = 20141231
END_BOUND = 20260101
PER_PAGE = 50
REQUEST_PAUSE_SECONDS = 2.2

## 2. GraphQL query

The query requests the fields needed for the later EDA and hypothesis test.

In [ ]:
QUERY = """
query (
  $page: Int!,
  $perPage: Int!,
  $source: MediaSource!,
  $startDate: FuzzyDateInt!,
  $endDate: FuzzyDateInt!
) {
  Page(page: $page, perPage: $perPage) {
    pageInfo { hasNextPage }
    media(
      type: ANIME
      format: TV
      status: FINISHED
      source: $source
      isAdult: false
      startDate_greater: $startDate
      startDate_lesser: $endDate
    ) {
      id
      idMal
      title { romaji english native }
      source
      format
      status
      startDate { year month day }
      endDate { year month day }
      season
      seasonYear
      episodes
      duration
      countryOfOrigin
      genres
      averageScore
      meanScore
      popularity
      favourites
      siteUrl
    }
  }
}
"""

## 3. API request helper

This function sends GraphQL POST requests and retries after rate-limit or temporary network errors.

In [ ]:
def post_graphql(query, variables, max_retries=6):
    payload = json.dumps({"query": query, "variables": variables}).encode("utf-8")
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    for attempt in range(max_retries):
        req = urllib.request.Request(API_URL, data=payload, headers=headers, method="POST")
        try:
            with urllib.request.urlopen(req, timeout=60) as response:
                result = json.loads(response.read().decode("utf-8"))
                if "errors" in result:
                    raise RuntimeError("AniList GraphQL error:\n" + json.dumps(result["errors"], indent=2))
                return result
        except urllib.error.HTTPError as e:
            if e.code == 429:
                retry_after = e.headers.get("Retry-After")
                try:
                    wait = int(retry_after)
                except (TypeError, ValueError):
                    wait = 60
                print(f"Rate limited. Waiting {wait} seconds...")
                time.sleep(wait)
                continue
            error_body = e.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"HTTP {e.code}\n{error_body[:1500]}") from e
        except urllib.error.URLError as e:
            if attempt < max_retries - 1:
                wait = 5 * (attempt + 1)
                print(f"Network error: {e}")
                print(f"Retrying in {wait} seconds...")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("Maximum retry attempts reached.")

## 4. Pagination and record-flattening helpers

In [ ]:
def safe_join(values):
    return " | ".join(values or [])

def fetch_source(source):
    rows = []
    page = 1
    while True:
        variables = {
            "page": page,
            "perPage": PER_PAGE,
            "source": source,
            "startDate": START_BOUND,
            "endDate": END_BOUND
        }
        result = post_graphql(QUERY, variables)
        page_data = result["data"]["Page"]
        media = page_data["media"]
        print(f"{source}: page {page}, {len(media)} records")
        for item in media:
            title = item.get("title") or {}
            start = item.get("startDate") or {}
            end = item.get("endDate") or {}
            rows.append({
                "anilist_id": item.get("id"),
                "mal_id": item.get("idMal"),
                "title_romaji": title.get("romaji"),
                "title_english": title.get("english"),
                "title_native": title.get("native"),
                "source": item.get("source"),
                "format": item.get("format"),
                "status": item.get("status"),
                "start_year": start.get("year"),
                "start_month": start.get("month"),
                "start_day": start.get("day"),
                "end_year": end.get("year"),
                "end_month": end.get("month"),
                "end_day": end.get("day"),
                "season": item.get("season"),
                "season_year": item.get("seasonYear"),
                "episodes": item.get("episodes"),
                "duration_min": item.get("duration"),
                "country_of_origin": item.get("countryOfOrigin"),
                "genres": safe_join(item.get("genres")),
                "average_score": item.get("averageScore"),
                "mean_score": item.get("meanScore"),
                "popularity": item.get("popularity"),
                "favourites": item.get("favourites"),
                "anilist_url": item.get("siteUrl"),
            })
        if not page_data["pageInfo"]["hasNextPage"]:
            break
        page += 1
        time.sleep(REQUEST_PAUSE_SECONDS)
    return rows

## 5. CSV and summary helpers

In [ ]:
def write_csv(filename, rows, fieldnames):
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def median(values):
    values = sorted(values)
    n = len(values)
    if n == 0:
        return None
    middle = n // 2
    if n % 2 == 1:
        return values[middle]
    return (values[middle - 1] + values[middle]) / 2

## 6. Download the data

This cell queries AniList separately for manga adaptations and light-novel adaptations, then combines them.

In [ ]:
print("Starting AniList download...\n")
all_rows = []
for source in ["MANGA", "LIGHT_NOVEL"]:
    all_rows.extend(fetch_source(source))
    time.sleep(REQUEST_PAUSE_SECONDS)
print(f"\nTotal records returned before deduplication: {len(all_rows):,}")

## 7. Remove duplicate AniList IDs

In [ ]:
deduped = {}
for row in all_rows:
    deduped[row["anilist_id"]] = row
raw_rows = list(deduped.values())
duplicate_count = len(all_rows) - len(raw_rows)
print(f"Duplicate AniList IDs removed: {duplicate_count:,}")
print(f"Raw rows after deduplication: {len(raw_rows):,}")

## 8. Save the raw dataset

In [ ]:
fieldnames = [
    "anilist_id", "mal_id", "title_romaji", "title_english", "title_native",
    "source", "format", "status", "start_year", "start_month", "start_day",
    "end_year", "end_month", "end_day", "season", "season_year", "episodes",
    "duration_min", "country_of_origin", "genres", "average_score", "mean_score",
    "popularity", "favourites", "anilist_url"
]
raw_file = "anilist_raw_manga_vs_light_novel_2015_2025.csv"
write_csv(raw_file, raw_rows, fieldnames)
print(f"Saved: {raw_file}")

## 9. Build the analysis dataset

The primary outcome is `average_score`, so only records missing that outcome are removed. No observations are dropped because their scores are unusually high or low.

In [ ]:
analysis_rows = [row.copy() for row in raw_rows if row["average_score"] is not None]
for row in analysis_rows:
    if row["source"] == "MANGA":
        row["source_group"] = "Manga adaptation"
    elif row["source"] == "LIGHT_NOVEL":
        row["source_group"] = "Light novel adaptation"
    else:
        row["source_group"] = row["source"]
analysis_fieldnames = fieldnames + ["source_group"]
analysis_file = "anilist_analysis_manga_vs_light_novel_2015_2025.csv"
write_csv(analysis_file, analysis_rows, analysis_fieldnames)
print(f"Saved: {analysis_file}")

## 10. Retrieval and cleaning summary

In [ ]:
raw_counts = {"MANGA": 0, "LIGHT_NOVEL": 0}
analysis_counts = {"Manga adaptation": 0, "Light novel adaptation": 0}
for row in raw_rows:
    if row["source"] in raw_counts:
        raw_counts[row["source"]] += 1
for row in analysis_rows:
    group = row["source_group"]
    if group in analysis_counts:
        analysis_counts[group] += 1
missing_scores = sum(1 for row in raw_rows if row["average_score"] is None)
print("================ RAW DATA ================")
print(f"Rows: {len(raw_rows):,}")
print(f"MANGA: {raw_counts['MANGA']:,}")
print(f"LIGHT_NOVEL: {raw_counts['LIGHT_NOVEL']:,}")
print(f"Duplicate AniList IDs removed: {duplicate_count:,}")
print(f"Missing average scores: {missing_scores:,}")
print("\n================ ANALYSIS DATA ================")
print(f"Rows: {len(analysis_rows):,}")
print(f"Manga adaptations: {analysis_counts['Manga adaptation']:,}")
print(f"Light novel adaptations: {analysis_counts['Light novel adaptation']:,}")
for group in ["Manga adaptation", "Light novel adaptation"]:
    scores = [float(row["average_score"]) for row in analysis_rows if row["source_group"] == group]
    if scores:
        mean_score = sum(scores) / len(scores)
        med_score = median(scores)
        print(f"{group}: n={len(scores)}, mean={mean_score:.2f}, median={med_score:.2f}, min={min(scores):.0f}, max={max(scores):.0f}")
years = [int(row["start_year"]) for row in analysis_rows if row["start_year"] is not None]
if years:
    print(f"\nRelease-year range: {min(years)} to {max(years)}")
print("\nSaved files:")
print(" -", raw_file)
print(" -", analysis_file)
print("\nDone.")

## Reproducibility note

Running this notebook from top to bottom re-creates both project CSV files directly from AniList. Because AniList is a live database, exact row counts may change if records are added or metadata are corrected in the future. The query criteria are documented above so the dataset can be reproduced and any changes can be explained.